# Dataset v1 · Resolution experiment training

Run the setup and configuration cells, then run either experiment cell independently. Each starts from the local COCO-pretrained YOLO11n checkpoint. Training uses the original 158 train / 17 validation / 15 test images and selects `best.pt` by Ultralytics validation fitness. Test images are reserved for notebook 08.

### Model V1 configuration audit

Notebook 05 launched YOLO11n at 640 px on CPU with 50 epochs, batch 4, zero workers and seed 42. Its initial Ultralytics 8.4.218 trainer log used `optimizer=auto`, `lr0=0.01` (ignored by auto), and `warmup_bias_lr=0.1` (auto sets it to 0). Auto selected AdamW with LR 0.002 and beta1 0.9. The resumed run recorded AdamW, LR 0.002, momentum/beta1 0.937 and warmup bias LR 0.0 in `args.yaml`. Both fresh experiments reproduce the initial recipe. The configuration preview below shows the full shared settings, including augmentation, loss and scheduler parameters.

EXP-01 (960) and EXP-02 (1280) use MPS. The CPU-trained V1 is a historical reference. EXP-01 versus EXP-02 is the comparison with a common training backend.

Batch 4 is fixed. At nominal square size, 960 and 1280 contain 2.25× and 4× as many input pixels as 640. Either run may exceed available unified memory. Treat a memory failure as a failed attempt and document any separately approved batch change.

In [1]:
import hashlib
import platform
import random
import shutil
import warnings
from tempfile import mkdtemp
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import ultralytics
import yaml
from IPython.display import display
from ultralytics import YOLO

root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "notebooks/05_model_v1_training.ipynb").is_file()
)
dataset_dir = root / "data/dataset_v1/yolo_dataset"
dataset_config = yaml.safe_load((dataset_dir / "data.yaml").read_text())
weights = root / "notebooks/yolo11n.pt"
runs = root / "data/dataset_v1/runs/resolution_experiments"
baseline_args_path = runs.parent / "exp_01_baseline/args.yaml"
experiments = {"EXP-01": 960, "EXP-02": 1280}
expected_counts = {"train": 158, "val": 17, "test": 15}

def sha256(path: Path) -> str:
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

In [2]:
# The resume log records these settings too, but the three optimizer inputs
# below come from the initial trainer log in notebook 05.
baseline_args = yaml.safe_load(baseline_args_path.read_text())
run_fields = {"model", "data", "project", "name", "save_dir", "mode", "resume"}
training_config = {key: value for key, value in baseline_args.items() if key not in run_fields}
training_config.update(optimizer="auto", lr0=0.01, warmup_bias_lr=0.1)
display(pd.Series(training_config, name="Model V1 initial trainer settings"))

task                        detect
epochs                          50
time                          None
patience                       100
batch                            4
                        ...       
copy_paste_mode               flip
auto_augment           randaugment
erasing                        0.4
cfg                           None
tracker            tracktrack.yaml
Name: Model V1 initial trainer settings, Length: 109, dtype: object

In [3]:
def train_experiment(name: str, *, retry_incomplete: bool = False) -> None:
    for other in experiments.keys() - {name}:
        manifest = runs / other / "experiment.yaml"
        if manifest.is_file():
            recorded_version = yaml.safe_load(manifest.read_text())["ultralytics"]
            if recorded_version != ultralytics.__version__:
                raise RuntimeError(f"Use Ultralytics {recorded_version} to match {other}.")
    if ultralytics.__version__ != "8.4.218":
        print(f"Model V1 used Ultralytics 8.4.218. This run uses {ultralytics.__version__}.")
    if platform.system() != "Darwin" or not torch.backends.mps.is_available():
        raise RuntimeError("This experiment requires macOS with PyTorch MPS.")
    if not weights.is_file():
        raise FileNotFoundError(weights)

    config = {**training_config, "imgsz": experiments[name], "device": "mps", "resume": False}
    paths = {
        split: sorted((dataset_dir / dataset_config[split]).glob("*.jpg"))
        for split in expected_counts
    }
    if {split: len(images) for split, images in paths.items()} != expected_counts:
        raise ValueError("Expected the original Dataset v1 split counts.")
    if dataset_config["names"] != {0: "license_plate"}:
        raise ValueError("Expected one license_plate class.")

    # Ultralytics writes label caches and may repair images in place.
    run_dir = runs / name
    if run_dir.exists():
        if (run_dir / "training_complete").exists():
            raise FileExistsError(f"Completed run already exists: {run_dir}")
        if not retry_incomplete:
            raise FileExistsError(f"Stop the previous attempt before retrying {run_dir}.")
        archive = Path(mkdtemp(prefix=f"{name}-incomplete-", dir=runs)) / name
        run_dir.rename(archive)
        print(f"Preserved previous attempt at {archive}")
    run_dir.mkdir(parents=True, exist_ok=False)
    local_dataset = run_dir / "dataset"
    for split, images in paths.items():
        image_dir = local_dataset / dataset_config[split]
        label_dir = local_dataset / "labels" / split
        image_dir.mkdir(parents=True)
        label_dir.mkdir(parents=True)
        for image in images:
            shutil.copy2(image, image_dir / image.name)
            label = dataset_dir / "labels" / split / image.with_suffix(".txt").name
            shutil.copy2(label, label_dir / label.name)

    data = local_dataset / "data.yaml"
    data.write_text(yaml.safe_dump({**dataset_config, "path": local_dataset.as_posix()}))
    provenance = {
        "training": config,
        "weights": weights.relative_to(root).as_posix(),
        "weights_sha256": sha256(weights),
        "dataset_counts": expected_counts,
        "ultralytics": ultralytics.__version__,
        "torch": str(torch.__version__),
    }
    (run_dir / "experiment.yaml").write_text(yaml.safe_dump(provenance, sort_keys=False))

    def check_batch(trainer) -> None:
        if trainer.batch_size != config["batch"] or trainer.train_loader.batch_size != config["batch"]:
            raise RuntimeError("Training batch changed. This attempt is invalid.")

    seed = config["seed"]
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    model = YOLO(str(weights))
    model.add_callback("on_train_batch_start", check_batch)
    model.add_callback("on_train_epoch_end", check_batch)
    with warnings.catch_warnings():
        if name == "EXP-02":
            warnings.filterwarnings(
                "ignore",
                message=r"^(scatter_reduce_mps|index_put_with_accumulate_mps) does not have a deterministic implementation",
                category=UserWarning,
                module=r"torch[.]autograd[.]graph$",
            )
        model.train(data=str(data), project=str(runs), name=name, **config)

    history = pd.read_csv(run_dir / "results.csv")
    if len(history) != config["epochs"] or not (run_dir / "weights/best.pt").is_file():
        raise RuntimeError("Training did not produce the expected complete run.")
    (run_dir / "training_complete").touch(exist_ok=False)
    display(history.tail())

## EXP-01 · 960 px

Set the flag to `True` to start this run. Each run directory is reserved once, so rerunning a cell cannot overwrite an existing attempt.

In [ ]:
run_exp_01 = False
if run_exp_01:
    train_experiment("EXP-01")

Model V1 used Ultralytics 8.4.218. This run uses 8.4.219.
Ultralytics 8.4.219 🚀 Python-3.12.16 torch-2.15.0 MPS (Apple M2)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/aarishtickoo/workspace/traffic-pii-masking/data/dataset_v1/runs/resolution_experiments/EXP-01/dataset/data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train

2026-11-26 19:15:46.717 Python[26838:11998528] Error creating directory 
 The volume ‚ÄúMacintosh HD‚Äù is out of space. You can‚Äôt save the file ‚Äúmpsgraph-26838-2026-11-26_19_15_46-2574022484‚Äù because the volume ‚ÄúMacintosh HD‚Äù is out of space.
2026-11-26 19:15:49.811 Python[26838:11998528] Error creating directory 
 The volume ‚ÄúMacintosh HD‚Äù is out of space. You can‚Äôt save the file ‚Äúmpsgraph-26838-2026-11-26_19_15_49-3082809117‚Äù because the volume ‚ÄúMacintosh HD‚Äù is out of space.



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       1/50       2.2G      1.102      22.71        1.2          6        960: 0% ──────────── 0/40  10.7s

/Users/aarishtickoo/workspace/traffic-pii-masking/.venv/lib/python3.12/site-packages/torch/autograd/graph.py:1113: UserWarning: scatter_reduce_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:224.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
/Users/aarishtickoo/workspace/traffic-pii-masking/.venv/lib/python3.12/site-packages/torch/autograd/graph.py:1113: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for thi

       1/50      3.25G      1.103      13.75      1.007          3        960: 100% ━━━━━━━━━━━━ 40/40 1.2it/s 33.2s0.6ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 1.1it/s 2.7s0.8s
                   all         17         15      0.726      0.709        0.7      0.438

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/50       3.2G     0.7449      8.905     0.8902          4        960: 0% ──────────── 0/40  0.7s

/Users/aarishtickoo/workspace/traffic-pii-masking/.venv/lib/python3.12/site-packages/torch/autograd/graph.py:1113: UserWarning: scatter_reduce_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:224.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
/Users/aarishtickoo/workspace/traffic-pii-masking/.venv/lib/python3.12/site-packages/torch/autograd/graph.py:1113: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for thi

... (94 outputs left out)

      49/50      3.23G     0.5801     0.4176     0.8397          1        960: 100% ━━━━━━━━━━━━ 40/40 1.0s/it 40.4s1.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 2.3it/s 1.3s0.6s
                   all         17         15      0.928      0.933      0.979       0.86

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      50/50      3.21G     0.5764     0.3523     0.8716          4        960: 0% ──────────── 0/40  1.8s

/Users/aarishtickoo/workspace/traffic-pii-masking/.venv/lib/python3.12/site-packages/torch/autograd/graph.py:1113: UserWarning: scatter_reduce_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:224.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
/Users/aarishtickoo/workspace/traffic-pii-masking/.venv/lib/python3.12/site-packages/torch/autograd/graph.py:1113: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for thi

      50/50      3.23G     0.5621     0.3905     0.8526          2        960: 100% ━━━━━━━━━━━━ 40/40 1.0s/it 40.3s0.9ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 2.7it/s 1.1s0.7s
                   all         17         15      0.928      0.933      0.979      0.872

50 epochs completed in 0.330 hours.
Optimizer stripped from /Users/aarishtickoo/workspace/traffic-pii-masking/data/dataset_v1/runs/resolution_experiments/EXP-01/weights/last.pt, 5.5MB
Optimizer stripped from /Users/aarishtickoo/workspace/traffic-pii-masking/data/dataset_v1/runs/resolution_experiments/EXP-01/weights/best.pt, 5.5MB

Validating /Users/aarishtickoo/workspace/traffic-pii-masking/data/dataset_v1/runs/resolution_experiments/EXP-01/weights/best.pt...
Ultralytics 8.4.219 🚀 Python-3.12.16 torch-2.15.0 MPS (Apple M2)
YOLO11n summary (fused): 101 layers, 2,597,051 parameters, 0 gradients, 6.4 GFLOPs
                 Class     Images  Instances   

,epoch,time,train/box_loss,train/cls_loss,train/dfl_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/box_loss,val/cls_loss,val/dfl_loss,lr/pg0,lr/pg1,lr/pg2
45,46,1019.61,0.60159,0.42438,0.85040,0.93259,0.92296,0.97221,0.83962,0.69735,0.83950,0.91647,0.000218,0.000218,0.000218
46,47,1056.57,0.56576,0.40596,0.81891,0.93048,0.93333,0.97221,0.84267,0.74023,0.77512,0.92209,0.000178,0.000178,0.000178
47,48,1100.09,0.56839,0.40174,0.84046,0.92956,0.93333,0.97864,0.86892,0.76541,0.73839,0.92824,0.000139,0.000139,0.000139
48,49,1143.39,0.58008,0.41763,0.83967,0.92776,0.93333,0.97864,0.86022,0.73707,0.73019,0.92414,0.000099,0.000099,0.000099
49,50,1186.21,0.56208,0.39047,0.85258,0.92757,0.93333,0.97864,0.87219,0.71584,0.73447,0.92004,0.000060,0.000060,0.000060


## EXP-02 · 1280 px

In [5]:
run_exp_02 = True
retry_exp_02 = True

if run_exp_02:
    train_experiment("EXP-02", retry_incomplete=retry_exp_02)

Model V1 used Ultralytics 8.4.218. This run uses 8.4.219.
Preserved previous attempt at /Users/aarishtickoo/workspace/traffic-pii-masking/data/dataset_v1/runs/resolution_experiments/EXP-02-incomplete-j8hwitrn/EXP-02
Ultralytics 8.4.219 🚀 Python-3.12.16 torch-2.15.0 MPS (Apple M2)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/aarishtickoo/workspace/traffic-pii-masking/data/dataset_v1/runs/resolution_experiments/EXP-02/dataset/data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze

,epoch,time,train/box_loss,train/cls_loss,train/dfl_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/box_loss,val/cls_loss,val/dfl_loss,lr/pg0,lr/pg1,lr/pg2
45,46,11297.1,0.54588,0.38945,0.82132,0.93696,0.99134,0.99567,0.84890,0.80177,0.52087,0.94003,0.000218,0.000218,0.000218
46,47,11471.5,0.53968,0.36966,0.82772,0.93490,1.00000,0.99134,0.83726,0.81493,0.52954,0.93580,0.000178,0.000178,0.000178
47,48,11689.6,0.55381,0.39265,0.85032,0.93443,1.00000,0.99134,0.83073,0.81664,0.52738,0.93280,0.000139,0.000139,0.000139
48,49,11815.6,0.53145,0.37959,0.84129,0.93036,1.00000,0.99134,0.83895,0.81292,0.57513,0.92970,0.000099,0.000099,0.000099
49,50,11946.4,0.53864,0.37425,0.84425,0.92869,1.00000,0.99134,0.82886,0.81420,0.58619,0.92915,0.000060,0.000060,0.000060
